# 03 - Cleaning, Feature Engineering, Split, and Feature Store

**Owners:** Rajneesh Kumar (cleaning, split) and Vinitha Palani (features, feature store)

This notebook:
1. Applies the cleaning rules in `src/preprocessing.py` to the training and testing files.
2. Builds features with `src/features.py` (row-wise, so nothing is fitted and no leakage occurs).
3. Splits the training file 80/20, stratified on churn, with a fixed seed.
4. Saves the splits to S3 and loads the features into two SageMaker Feature Store feature groups.

Scaling and one-hot encoding are done later inside the training pipeline (notebook 04), fitted on the training split only.

In [1]:
import sys, time
sys.path.append("..")

import pandas as pd
from sklearn.model_selection import train_test_split

from src import config, data_utils, preprocessing, features

pd.set_option("display.max_columns", 40)

## 1. Clean both files

In [2]:
train_raw = data_utils.load_raw("train")
test_raw = data_utils.load_raw("test")

train_clean, train_report = preprocessing.clean(train_raw)
test_clean, test_report = preprocessing.clean(test_raw)

print("train cleaning report:", train_report)
print("test  cleaning report:", test_report)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


[10/19/26 07:19:30] INFO     Initializing a Ray instance                                               ]8;id=5214753;file:///opt/conda/lib/python3.12/site-packages/awswrangler/distributed/ray/_core.py\_core.py]8;;\:]8;id=5214754;file:///opt/conda/lib/python3.12/site-packages/awswrangler/distributed/ray/_core.py#202\202]8;;\

2026-10-19 07:19:32,401	WARNING services.py:2213 -- WARNING: The object store is using /tmp/ray instead of /dev/shm because /dev/shm has only 906993664 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=2.18gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2026-10-19 07:19:35,566	INFO worker.py:2012 -- Started a local Ray instance.
/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


train cleaning report: {'rows_in': 163284, 'dropped_missing_label_or_id': 1, 'dropped_duplicates': 0, 'rows_with_null_features': 0, 'null_rule': 'dropped', 'dropped_out_of_range': 0, 'rows_out': 163283}
test  cleaning report: {'rows_in': 64374, 'dropped_missing_label_or_id': 0, 'dropped_duplicates': 0, 'rows_with_null_features': 0, 'null_rule': 'dropped', 'dropped_out_of_range': 0, 'rows_out': 64374}


## 2. Engineer features

In [3]:
train_fe = features.engineer_features(train_clean)
prod_fe = features.engineer_features(test_clean)     # the testing file is the unseen "production" data

print("model features:", features.MODEL_FEATURES)
print("excluded from the model:", features.EXCLUDED_FROM_MODEL)
train_fe.head()

model features: ['age', 'tenure', 'usage_frequency', 'support_calls', 'payment_delay', 'last_interaction', 'total_spend', 'spend_per_month', 'calls_per_month', 'engagement_ratio', 'subscription_type', 'contract_length', 'tenure_bucket', 'monthly_contract', 'high_payment_delay', 'many_support_calls']
excluded from the model: ['customer_id', 'churn', 'gender']


,customer_id,age,gender,tenure,usage_frequency,support_calls,payment_delay,subscription_type,contract_length,total_spend,last_interaction,churn,tenure_bucket,spend_per_month,calls_per_month,engagement_ratio,high_payment_delay,many_support_calls,monthly_contract
0,2,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1,37-60,23.897436,0.128205,0.823529,1,1,0
1,3,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1,37-60,11.367347,0.204082,0.166667,0,1,1
2,4,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1,13-24,13.214286,0.428571,1.333333,1,1,0
3,5,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1,37-60,10.421053,0.184211,0.724138,0,1,1
4,6,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1,25-36,19.281250,0.156250,1.000000,0,1,1


In [4]:
# Sanity checks on the new features
print(train_fe[features.NUMERIC_FEATURES].describe().T[["min", "mean", "max"]])
print("\nnulls in model features:", int(train_fe[features.MODEL_FEATURES].isna().sum().sum()))
print(train_fe.groupby("tenure_bucket")["churn"].agg(["mean", "count"]))

                         min        mean     max
age                18.000000   41.678442    65.0
tenure              1.000000   30.510133    60.0
usage_frequency     1.000000   15.492868    30.0
support_calls       0.000000    5.102932    10.0
payment_delay       0.000000   15.151461    30.0
last_interaction    1.000000   15.564701    30.0
total_spend       100.000000  544.499287  1000.0
spend_per_month     1.666667   42.601369  1000.0
calls_per_month     0.000000    0.397458    10.0
engagement_ratio    0.033333    2.046096    30.0

nulls in model features: 0
                   mean  count
tenure_bucket                 
0-12           0.987031  32616
13-24          0.990941  32897
25-36          0.985272  32388
37-60          0.985638  65382


## 3. Split the training file 80/20

Stratified on `churn` so both splits keep the same churn rate. Hyperparameter tuning and the
decision threshold use cross-validation inside the 80% training split; the 20% test split is used
once at the end.

In [5]:
train_df, test_df = train_test_split(
    train_fe, test_size=config.TEST_SIZE, stratify=train_fe["churn"], random_state=config.RANDOM_SEED
)
for name, part in [("train (80%)", train_df), ("test (20%)", test_df), ("production file", prod_fe)]:
    print(f"{name:16s} rows={len(part):>8,}  churn rate={part['churn'].mean():.2%}")

train (80%)      rows= 130,626  churn rate=98.69%
test (20%)       rows=  32,657  churn rate=98.69%
production file  rows=  64,374  churn rate=47.37%


## 4. Save the processed data to S3

In [6]:
import awswrangler as wr
from sagemaker.core.helper.session_helper import Session

sagemaker_session = Session()
BUCKET = sagemaker_session.default_bucket()
outputs = {"train": train_df, "test": test_df, "production": prod_fe}
for name, frame in outputs.items():
    path = f"s3://{BUCKET}/{config.PROCESSED_PREFIX}/{name}/{name}.parquet"
    wr.s3.to_parquet(frame, path=path, index=False)
    print("saved", path)

saved s3://sagemaker-us-east-1-025824828736/churn/processed/train/train.parquet
saved s3://sagemaker-us-east-1-025824828736/churn/processed/test/test.parquet
saved s3://sagemaker-us-east-1-025824828736/churn/processed/production/production.parquet


## 5. Design the feature groups

Two feature groups, keyed by `customer_id`:

| Feature group | Contents |
|---|---|
| `churn-customer-profile` | age, tenure, subscription type, contract length, tenure bucket, monthly-contract flag |
| `churn-customer-activity` | usage frequency, support calls, payment delay, last interaction, total spend, and the engineered ratios and flags |

`gender` is deliberately **not** stored, because it is excluded from the model; it stays in the S3 splits for the bias audit.
Only the training file (train + test splits) is ingested; the production file is kept in S3 because customer IDs may overlap between the two Kaggle files.

In [7]:
import boto3
from botocore.exceptions import ClientError
from pandas.api.types import is_bool_dtype, is_float_dtype, is_integer_dtype
from sagemaker.core.helper.session_helper import Session, get_execution_role

sagemaker_session = Session()
role = get_execution_role()
region = sagemaker_session.boto_region_name
sm_client = boto3.client("sagemaker", region_name=region)
FS_S3_URI = f"s3://{BUCKET}/churn/feature-store"

FEATURE_GROUPS = {
    "churn-customer-profile": features.PROFILE_FEATURES,
    "churn-customer-activity": features.ACTIVITY_FEATURES,
}
all_train_file = pd.concat([train_df, test_df], ignore_index=True)
event_time = time.time()

def feature_store_type(dtype):
    if is_bool_dtype(dtype) or is_integer_dtype(dtype):
        return "Integral"
    if is_float_dtype(dtype):
        return "Fractional"
    return "String"

def create_feature_group(name, columns):
    frame = features.to_feature_store_frame(all_train_file, columns, event_time)
    definitions = [
        {"FeatureName": column, "FeatureType": feature_store_type(frame[column].dtype)}
        for column in frame.columns
    ]
    try:
        sm_client.create_feature_group(
            FeatureGroupName=name,
            RecordIdentifierFeatureName="customer_id",
            EventTimeFeatureName="event_time",
            FeatureDefinitions=definitions,
            OnlineStoreConfig={"EnableOnlineStore": True},
            OfflineStoreConfig={"S3StorageConfig": {"S3Uri": FS_S3_URI}},
            RoleArn=role,
        )
        print(f"creating {name}")
    except ClientError as err:
        if err.response["Error"]["Code"] not in {"ResourceInUse", "ResourceInUseException"}:
            raise
        print(f"{name} already exists")
    while True:
        description = sm_client.describe_feature_group(FeatureGroupName=name)
        status = description["FeatureGroupStatus"]
        if status != "Creating":
            break
        time.sleep(5)
    print(name, "->", status)
    if status != "Created":
        raise RuntimeError(f"{name} ended in {status}: {description.get('FailureReason', 'unknown')}")
    return description, frame

groups = {name: create_feature_group(name, cols) for name, cols in FEATURE_GROUPS.items()}

churn-customer-profile already exists
churn-customer-profile -> Created
churn-customer-activity already exists
churn-customer-activity -> Created


In [8]:
# Batch ingestion (up to 25 records per API request) with progress and retries.
import math

runtime = boto3.client("sagemaker-featurestore-runtime", region_name=region)

def format_feature_value(value, feature_type):
    if feature_type == "Integral":
        return str(int(value))
    if feature_type == "Fractional":
        return str(float(value))
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return str(value)

def make_record(row, feature_types):
    record = [
        {"FeatureName": column, "ValueAsString": format_feature_value(value, feature_types[column])}
        for column, value in row.items() if not pd.isna(value)
    ]
    included = {item["FeatureName"] for item in record}
    missing = {"customer_id", "event_time"} - included
    if missing:
        raise ValueError(f"record is missing required features: {sorted(missing)}")
    return record

def ingest_frame(name, description, frame, batch_size=25):
    feature_types = {f["FeatureName"]: f["FeatureType"] for f in description["FeatureDefinitions"]}
    total = len(frame)
    batches = math.ceil(total / batch_size)
    for batch_number, start in enumerate(range(0, total, batch_size), start=1):
        part = frame.iloc[start:start + batch_size]
        pending = [
            {
                "FeatureGroupName": name,
                "Record": make_record(row, feature_types),
                "TargetStores": ["OnlineStore", "OfflineStore"],
            }
            for _, row in part.iterrows()
        ]
        for attempt in range(1, 6):
            response = runtime.batch_write_record(Entries=pending)
            errors = response.get("Errors", [])
            if errors:
                details = "; ".join(f"{e['ErrorCode']}: {e['ErrorMessage']}" for e in errors)
                raise RuntimeError(f"batch ingestion failed: {details}")
            pending = response.get("UnprocessedEntries", [])
            if not pending:
                break
            time.sleep(min(2 ** attempt, 10))
        else:
            raise RuntimeError(f"{len(pending)} records remained unprocessed after five attempts")
        completed = min(start + batch_size, total)
        print(f"{name}: {completed}/{total} records ({batch_number}/{batches} batches)")
    print(f"finished ingesting {total} records into {name}")

for name, (description, frame) in groups.items():
    ingest_frame(name, description, frame)

churn-customer-profile: 25/163283 records (1/6532 batches)
churn-customer-profile: 50/163283 records (2/6532 batches)
churn-customer-profile: 75/163283 records (3/6532 batches)
churn-customer-profile: 100/163283 records (4/6532 batches)
churn-customer-profile: 125/163283 records (5/6532 batches)
churn-customer-profile: 150/163283 records (6/6532 batches)
churn-customer-profile: 175/163283 records (7/6532 batches)
churn-customer-profile: 200/163283 records (8/6532 batches)
churn-customer-profile: 225/163283 records (9/6532 batches)
churn-customer-profile: 250/163283 records (10/6532 batches)
churn-customer-profile: 275/163283 records (11/6532 batches)
churn-customer-profile: 300/163283 records (12/6532 batches)
churn-customer-profile: 325/163283 records (13/6532 batches)
churn-customer-profile: 350/163283 records (14/6532 batches)
churn-customer-profile: 375/163283 records (15/6532 batches)
churn-customer-profile: 400/163283 records (16/6532 batches)
churn-customer-profile: 425/163283 r

## 6. Verify: read one customer back from the online store

In [9]:
runtime = boto3.client("sagemaker-featurestore-runtime", region_name=region)
sample_id = str(train_df["customer_id"].iloc[0])
for name in FEATURE_GROUPS:
    record = runtime.get_record(FeatureGroupName=name, RecordIdentifierValueAsString=sample_id)
    print(name, {r["FeatureName"]: r["ValueAsString"] for r in record["Record"]})

churn-customer-profile {'customer_id': '9457', 'age': '59.0', 'tenure': '7.0', 'subscription_type': 'Basic', 'contract_length': 'Quarterly', 'tenure_bucket': '0-12', 'monthly_contract': '0', 'event_time': '1792394400.88521'}
churn-customer-activity {'customer_id': '9457', 'usage_frequency': '5.0', 'support_calls': '3.0', 'payment_delay': '9.0', 'last_interaction': '2.0', 'total_spend': '331.0', 'spend_per_month': '47.285714285714285', 'calls_per_month': '0.42857142857142855', 'engagement_ratio': '2.5', 'high_payment_delay': '0', 'many_support_calls': '0', 'event_time': '1792394400.88521'}


In [10]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>


I0000 00:00:1792395446.773782    4910 chttp2_transport.cc:1182] ipv4:169.255.255.2:37703: Got goaway [2] err=UNAVAILABLE:GOAWAY received; Error code: 2; Debug Text: Cancelling all calls {grpc_status:14, http2_error:2, created_time:"2026-10-19T07:37:26.773777738+00:00"}
*** SIGTERM received at time=1792395449 on cpu 1 ***
PC: @     0x7f16b9b9e2d2  (unknown)  epoll_wait
[2026-10-19 07:37:29,251 E 4613 4613] logging.cc:474: *** SIGTERM received at time=1792395449 on cpu 1 ***
[2026-10-19 07:37:29,251 E 4613 4613] logging.cc:474: PC: @     0x7f16b9b9e2d2  (unknown)  epoll_wait
